# Terra Lens — Pipeline A: Full Dual-Domain Training on S2-NAIP 5k

**Team TechTonic.** This notebook trains the *full* Terra Lens architecture end-to-end  
on the [S2-NAIP 5k pairs (RGB + NIR)](https://www.kaggle.com/datasets/evgeniyivankin/s2-naip-5k-pairs-rgb-nir) dataset.

**Architecture:**  
Edge-guided RSTB spatial branch (windowed attention) + FFT-based frequency branch (FRN)  
+ AC-energy guided fusion + PixelShuffle upsampler — with global bicubic residual.

**Two-stage cascade:**  
1. **Stage 1 (real pairs):** Train the full network on real S2 / NAIP pairs at x4.  
2. **Stage 2 (fine-tune):** Lower LR, longer warmup, further refinement.

**Kaggle Optimizations (T4x2, 30 GB total VRAM, 12h session, ~13 GB RAM):**
- Windowed self-attention (8x8 windows) instead of global — O(N*w^2) vs O(N^2)
- Mixed-precision (AMP fp16) throughout
- Gradient checkpointing on RSTB blocks
- DataParallel across both T4 GPUs
- Gradient accumulation (effective batch = 32)
- Memory-mapped / lazy .npy loading
- Aggressive `torch.cuda.empty_cache()` + `gc.collect()`
- Early stopping to avoid wasting session time

In [ ]:
# ============================================================
# 0. Environment setup — Kaggle GPU kernel (Settings -> Accelerator -> GPU T4x2)
# ============================================================
import os, sys, math, glob, random, json, time, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, Subset
from pathlib import Path

# Must be set BEFORE CUDA init
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")
else:
    n_gpus = 0
    print("  WARNING: No GPU detected -- training will be extremely slow on CPU.")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = torch.cuda.is_available()
AMP_DTYPE = torch.float16  # T4 does not support bf16
USE_AMP = USE_CUDA

## 1. Dataset Discovery & Extraction -- S2-NAIP 5k pairs

The S2-NAIP 5k dataset (from `evgeniyivankin/s2-naip-5k-pairs-rgb-nir`) contains:  
- **`naip/`** -- High-resolution NAIP imagery (~1.25 m/pixel), PNG files with 4 channels (RGB + NIR)  
- **`sentinel2/`** -- Low-resolution Sentinel-2 imagery (~10 m/pixel), TIF files with 4 channels  

Files are matched by **Tile ID** (e.g., `32612_960_-6049`).  
The effective super-resolution scale is ~8x, but we train at **4x bicubic downscale** of NAIP for  
a well-conditioned problem (LR input = NAIP downscaled 4x, HR target = NAIP native).

**Why use NAIP as both LR source and HR target (Wald-style)?**  
The real S2 and NAIP pairs have different acquisition times, atmospheres, and spectral responses,  
so directly training S2 to NAIP is a cross-sensor transfer problem (harder, noisier).  
Instead, we also offer a mode to use real S2 as input and NAIP as target for domain adaptation.

In [ ]:
# ============================================================
# 1a. Locate and inspect the dataset
# ============================================================

# Kaggle mounts datasets read-only at /kaggle/input/<dataset-slug>/
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")

# Auto-detect the dataset root
def find_dataset_root():
    """Search common Kaggle mount points for the S2-NAIP dataset."""
    candidates = [
        KAGGLE_INPUT / "s2-naip-5k-pairs-rgb-nir",
        KAGGLE_INPUT / "s2naip5kpairsrgbnir",
        KAGGLE_INPUT / "datasets/evgeniyivankin/s2-naip-5k-pairs-rgb-nir/train",
        Path("./s2_naip_dataset"),
    ]
    # Also search any subdir of /kaggle/input/ that has naip or sentinel2 folders
    if KAGGLE_INPUT.exists():
        for d in KAGGLE_INPUT.iterdir():
            if d.is_dir():
                candidates.append(d)
    
    for c in candidates:
        if not c.exists():
            continue
        # Check if it directly has naip/ and sentinel2/
        if (c / "naip").is_dir() or (c / "sentinel2").is_dir():
            return c
        # Check one level deeper (some datasets have an extra wrapper dir)
        for sub in c.iterdir():
            if sub.is_dir() and ((sub / "naip").is_dir() or (sub / "sentinel2").is_dir()):
                return sub
        # Check if it has .npy / .png / .tif files directly (flat layout)
        exts = set()
        for p in c.rglob("*"):
            if p.is_file():
                exts.add(p.suffix.lower())
            if len(exts) > 10:
                break
        if exts & {".npy", ".png", ".tif", ".tiff"}:
            return c
    
    raise FileNotFoundError(
        "Could not find the S2-NAIP dataset. Make sure you added it via:\n"
        "  Notebook -> Add Data -> search 's2-naip-5k-pairs-rgb-nir'\n"
        "Available dirs under /kaggle/input/: "
        + str(list(KAGGLE_INPUT.iterdir()) if KAGGLE_INPUT.exists() else "N/A")
    )

DATA_ROOT = find_dataset_root()
print(f"Dataset root: {DATA_ROOT}")

# List top-level contents
for item in sorted(DATA_ROOT.iterdir()):
    if item.is_dir():
        n_files = sum(1 for _ in item.rglob("*") if _.is_file())
        print(f"  [DIR] {item.name}/  ({n_files} files)")
    else:
        print(f"  [FILE] {item.name}  ({item.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# ============================================================
# 1b. Build paired file lists (S2 <-> NAIP matched by Tile ID)
# ============================================================
IMG_EXTS = (".npy", ".tif", ".tiff", ".png", ".jpg", ".jpeg")

def discover_files(root_dir):
    """
    Discover all image files and organize into HR (NAIP) and LR (Sentinel-2) lists.
    Handles multiple dataset layouts:
      1. naip/ + sentinel2/ subfolders (standard S2-NAIP layout)
      2. Flat directory with naming conventions
      3. HR-only (will synthesize LR via bicubic downscale)
    """
    root = Path(root_dir)
    
    # ---- Layout 1: Separate naip/ and sentinel2/ folders ----
    naip_dir = None
    s2_dir = None
    for d in root.rglob("*"):
        if d.is_dir():
            name_lower = d.name.lower()
            if name_lower == "naip" and naip_dir is None:
                naip_dir = d
            elif name_lower in ("sentinel2", "s2", "sentinel-2") and s2_dir is None:
                s2_dir = d
    
    if naip_dir and naip_dir.exists():
        hr_files = sorted([f for f in naip_dir.rglob("*") 
                          if f.is_file() and f.suffix.lower() in IMG_EXTS])
        print(f"Found {len(hr_files)} NAIP (HR) files in {naip_dir}")
        
        if s2_dir and s2_dir.exists():
            lr_files = sorted([f for f in s2_dir.rglob("*")
                              if f.is_file() and f.suffix.lower() in IMG_EXTS])
            print(f"Found {len(lr_files)} Sentinel-2 (LR) files in {s2_dir}")
            return hr_files, lr_files, "paired"
        else:
            print("No Sentinel-2 folder found -- will synthesize LR from NAIP via bicubic downscale.")
            return hr_files, [], "hr_only"
    
    # ---- Layout 2: Flat directory with all files ----
    all_files = sorted([f for f in root.rglob("*")
                       if f.is_file() and f.suffix.lower() in IMG_EXTS])
    
    # Try to split by filename convention
    naip_files = [f for f in all_files if "naip" in f.stem.lower() or "hr" in f.stem.lower()]
    s2_files = [f for f in all_files if "s2" in f.stem.lower() or "sentinel" in f.stem.lower() or "lr" in f.stem.lower()]
    
    if naip_files:
        print(f"Found {len(naip_files)} NAIP files, {len(s2_files)} S2 files (flat layout)")
        if s2_files:
            return naip_files, s2_files, "paired"
        return naip_files, [], "hr_only"
    
    # Fallback: treat all files as HR
    print(f"Found {len(all_files)} total image files -- treating all as HR targets.")
    return all_files, [], "hr_only"


def extract_tile_id(filepath):
    """Extract the tile ID from a filename for S2<->NAIP matching.
    E.g., '32612_960_-6049.png' -> '32612_960_-6049'
         '32612_960_-6049_16.tif' -> '32612_960_-6049'
    """
    stem = Path(filepath).stem
    parts = stem.split("_")
    if len(parts) >= 3:
        return "_".join(parts[:3])
    return stem


hr_files, lr_files, layout_mode = discover_files(DATA_ROOT)
print(f"\nLayout mode: {layout_mode}")
print(f"HR files: {len(hr_files)} | LR files: {len(lr_files)}")

# Show samples
if hr_files:
    print(f"\nSample HR files: {[f.name for f in hr_files[:3]]}")
if lr_files:
    print(f"Sample LR files: {[f.name for f in lr_files[:3]]}")

In [ ]:
# ============================================================
# 1c. Image I/O -- handles .npy, .tif, .png with auto-normalization
# ============================================================

def read_image(path):
    """Load an image and return float32 array (4, H, W) in [0, 1].
    
    Handles:
      - .npy files (expected shape [4, H, W] or [H, W, 4])
      - .tif/.tiff (GeoTIFF via tifffile, 4-band)
      - .png/.jpg (RGB -> duplicate R as NIR proxy, or RGBA as 4-band)
    """
    path = str(path)
    ext = os.path.splitext(path)[1].lower()
    
    if ext == ".npy":
        arr = np.load(path)
    elif ext in (".tif", ".tiff"):
        try:
            import tifffile
            arr = tifffile.imread(path)
        except ImportError:
            try:
                import rasterio
                with rasterio.open(path) as src:
                    arr = src.read()  # (C, H, W)
            except ImportError:
                raise ImportError("Install tifffile or rasterio to read .tif files")
    else:
        from PIL import Image
        pil = Image.open(path)
        if pil.mode == "RGBA":
            arr = np.array(pil)  # (H, W, 4) -- treat A channel as NIR
        elif pil.mode == "RGB":
            arr = np.array(pil)  # (H, W, 3)
        else:
            arr = np.array(pil.convert("RGB"))  # (H, W, 3)
    
    # Ensure 3D
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]  # (1, H, W)
    
    # Convert HWC -> CHW if needed
    if arr.ndim == 3 and arr.shape[0] not in (1, 3, 4) and arr.shape[-1] in (3, 4):
        arr = arr.transpose(2, 0, 1)  # HWC -> CHW
    
    # Handle 3-channel (RGB) -> 4-channel (RGB + NIR proxy)
    if arr.shape[0] == 3:
        nir = arr[0:1].copy()  # Red -> NIR proxy
        arr = np.concatenate([arr, nir], axis=0)  # (4, H, W)
    elif arr.shape[0] != 4:
        if arr.shape[0] > 4:
            arr = arr[:4]
        else:
            pad = np.repeat(arr[-1:], 4 - arr.shape[0], axis=0)
            arr = np.concatenate([arr, pad], axis=0)
    
    # Normalize to [0, 1]
    arr = arr.astype(np.float32)
    if arr.max() > 255.5:
        arr = arr / 10000.0  # Sentinel-2 DN
    elif arr.max() > 1.5:
        arr = arr / 255.0  # Standard 8-bit
    
    return np.clip(arr, 0.0, 1.0)


# Quick test: read one sample
if hr_files:
    sample = read_image(hr_files[0])
    print(f"Sample HR image: shape={sample.shape}, range=[{sample.min():.3f}, {sample.max():.3f}]")
    print(f"  Per-band means: {[f'{sample[i].mean():.3f}' for i in range(4)]}")
    del sample; gc.collect()

In [ ]:
# ============================================================
# 1d. Dataset class -- efficient, Kaggle-optimized
# ============================================================

SCALE = 4  # S2 (~10m) / NAIP (~1.25m) ~ 8x but we train at 4x for tractability
PATCH_SIZE = 128  # HR patch size (LR patch = 128/4 = 32)


class S2NAIPDataset(Dataset):
    """Efficient dataset for S2-NAIP super-resolution training.
    
    Two modes:
      - 'paired': real S2 (LR) + NAIP (HR) pairs
      - 'hr_only': NAIP only, synthesize LR via bicubic downscale (Wald-style)
    
    Memory optimization: loads images lazily, one at a time.
    """
    def __init__(self, hr_files, lr_files=None, patch_size=128, scale=4,
                 is_train=True, mode="hr_only"):
        self.hr_files = hr_files
        self.lr_files = lr_files or []
        self.patch_size = patch_size
        self.scale = scale
        self.is_train = is_train
        self.mode = mode
        
        # Build tile-ID-based pairing if we have both
        if mode == "paired" and lr_files:
            hr_by_tile = {extract_tile_id(f): f for f in hr_files}
            lr_by_tile = {extract_tile_id(f): f for f in lr_files}
            common_tiles = sorted(set(hr_by_tile.keys()) & set(lr_by_tile.keys()))
            if common_tiles:
                self.pairs = [(hr_by_tile[t], lr_by_tile[t]) for t in common_tiles]
                print(f"  Matched {len(self.pairs)} paired tiles by ID")
            else:
                n = min(len(hr_files), len(lr_files))
                self.pairs = list(zip(hr_files[:n], lr_files[:n]))
                print(f"  No tile-ID matches; paired {n} files by index")
        else:
            self.pairs = [(f, None) for f in hr_files]
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        hr_path, lr_path = self.pairs[idx]
        
        # Load HR image
        hr = read_image(hr_path)  # (4, H, W), float32, [0, 1]
        _, h, w = hr.shape
        ps = self.patch_size
        
        # Pad if too small
        if h < ps or w < ps:
            hr = np.pad(hr, ((0, 0), (0, max(0, ps - h)), (0, max(0, ps - w))), mode="edge")
            _, h, w = hr.shape
        
        # Random or center crop
        if self.is_train:
            y = random.randint(0, h - ps)
            x = random.randint(0, w - ps)
        else:
            y = (h - ps) // 2
            x = (w - ps) // 2
        
        hr_patch = hr[:, y:y+ps, x:x+ps].copy()
        hr_t = torch.from_numpy(hr_patch).float()
        
        # Generate or load LR
        if self.mode == "paired" and lr_path is not None:
            lr = read_image(lr_path)  # (4, Hl, Wl)
            lr_ps = ps // self.scale
            _, lh, lw = lr.shape
            if lh < lr_ps or lw < lr_ps:
                lr = np.pad(lr, ((0, 0), (0, max(0, lr_ps - lh)), (0, max(0, lr_ps - lw))), mode="edge")
                _, lh, lw = lr.shape
            ly = min(y // self.scale, max(0, lh - lr_ps))
            lx = min(x // self.scale, max(0, lw - lr_ps))
            lr_patch = lr[:, ly:ly+lr_ps, lx:lx+lr_ps].copy()
            lr_t = torch.from_numpy(lr_patch).float()
        else:
            # Synthesize LR from HR via bicubic downscale (anti-aliased)
            lr_t = F.interpolate(
                hr_t.unsqueeze(0), scale_factor=1.0 / self.scale,
                mode='bicubic', align_corners=False, antialias=True
            ).squeeze(0).clamp(0, 1)
        
        # Data augmentation (training only)
        if self.is_train:
            if random.random() < 0.5:
                hr_t = hr_t.flip(-1)
                lr_t = lr_t.flip(-1)
            if random.random() < 0.5:
                hr_t = hr_t.flip(-2)
                lr_t = lr_t.flip(-2)
            k = random.randint(0, 3)
            if k > 0:
                hr_t = torch.rot90(hr_t, k, dims=(-2, -1))
                lr_t = torch.rot90(lr_t, k, dims=(-2, -1))
        
        return lr_t, hr_t


print(f"\nScale factor: {SCALE}x")
print(f"HR patch size: {PATCH_SIZE} -> LR patch size: {PATCH_SIZE // SCALE}")

In [ ]:
# ============================================================
# 1e. Create train/val split (reproducible, by file index)
# ============================================================

n_total = len(hr_files)
perm = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED)).tolist()

n_train = int(0.85 * n_total)
train_indices = perm[:n_train]
val_indices = perm[n_train:]

train_hr = [hr_files[i] for i in train_indices]
val_hr = [hr_files[i] for i in val_indices]

if layout_mode == "paired" and lr_files:
    train_lr = [lr_files[i] if i < len(lr_files) else None for i in train_indices]
    val_lr = [lr_files[i] if i < len(lr_files) else None for i in val_indices]
else:
    train_lr, val_lr = [], []

train_ds = S2NAIPDataset(train_hr, train_lr, patch_size=PATCH_SIZE, scale=SCALE,
                         is_train=True, mode=layout_mode)
val_ds = S2NAIPDataset(val_hr, val_lr, patch_size=PATCH_SIZE, scale=SCALE,
                       is_train=False, mode=layout_mode)

print(f"\nTrain: {len(train_ds)} samples | Val: {len(val_ds)} samples")

# Dataloader config -- optimized for Kaggle
BATCH_SIZE = 8
ACCUM_STEPS = 2  # gradient accumulation -> effective batch = 16 (x2 GPU = 32)
NUM_WORKERS = 2  # Kaggle kernels have 4 CPUs total

loader_kw = dict(
    num_workers=NUM_WORKERS,
    pin_memory=USE_CUDA,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)

print(f"Batch size: {BATCH_SIZE} x accum {ACCUM_STEPS} = effective {BATCH_SIZE * ACCUM_STEPS}")
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")

## 2. Terra Lens Architecture -- Kaggle-Optimized

Key modifications for T4x2 (15 GB VRAM each):
- **Windowed self-attention** (8x8 windows) replaces global attention -> O(N * 64) vs O(N^2)
- **Gradient checkpointing** on RSTB blocks to trade compute for memory
- **dim=48** (reduced from 60) to fit comfortably in VRAM
- **FFT-based FRN** (no DCT matrix materialization -- uses `torch.fft.rfft2`)
- **PixelShuffle x4** upsampler

In [ ]:
# ============================================================
# 2a. Sobel edge map -- fixed kernels, no gradients needed
# ============================================================

class SobelEdge(nn.Module):
    """Compute Sobel gradient magnitude -> normalized edge map in [0, 1]."""
    def __init__(self, n_bands=4):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
        ky = kx.t()
        self.register_buffer("kx", kx.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.register_buffer("ky", ky.view(1, 1, 3, 3).repeat(n_bands, 1, 1, 1))
        self.n_bands = n_bands

    @torch.no_grad()
    def forward(self, x):
        x = x.float()
        xp = F.pad(x, [1, 1, 1, 1], mode="reflect")
        gx = F.conv2d(xp, self.kx, groups=self.n_bands)
        gy = F.conv2d(xp, self.ky, groups=self.n_bands)
        mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-6).mean(dim=1, keepdim=True)
        mag = mag / (mag.mean(dim=(2, 3), keepdim=True) + 1e-6)
        return torch.tanh(mag)

In [ ]:
# ============================================================
# 2b. Edge-guided windowed self-attention RSTB
#     Uses 8x8 windows -> O(N x 64) instead of O(N^2)
#     + learnable edge bias on attention logits
# ============================================================

def window_partition(x, win):
    """Partition (B, C, H, W) into windows -> (B*nW, C, win, win)."""
    B, C, H, W = x.shape
    x = x.view(B, C, H // win, win, W // win, win)
    x = x.permute(0, 2, 4, 1, 3, 5).contiguous().view(-1, C, win, win)
    return x


def window_reverse(windows, win, H, W, B):
    """Reverse window_partition: (B*nW, C, win, win) -> (B, C, H, W)."""
    C = windows.shape[1]
    nH, nW = H // win, W // win
    x = windows.view(B, nH, nW, C, win, win)
    x = x.permute(0, 3, 1, 4, 2, 5).contiguous().view(B, C, H, W)
    return x


class WindowedEdgeAttention(nn.Module):
    """Multi-head self-attention within 8x8 windows, with edge-map bias."""
    def __init__(self, dim, window_size=8, num_heads=6):
        super().__init__()
        self.dim = dim
        self.win = window_size
        self.heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        
        # Relative position bias
        self.rel_pos = nn.Parameter(torch.zeros((2 * window_size - 1) ** 2, num_heads))
        nn.init.trunc_normal_(self.rel_pos, std=0.02)
        
        coords = torch.stack(torch.meshgrid(
            torch.arange(window_size), torch.arange(window_size), indexing="ij"
        )).flatten(1)
        rel = coords[:, :, None] - coords[:, None, :]
        rel = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += window_size - 1
        rel[:, :, 1] += window_size - 1
        rel[:, :, 0] *= 2 * window_size - 1
        self.register_buffer("rel_index", rel.sum(-1))
        
        # Edge-guided attention bias (zero-init)
        self.edge_beta = nn.Parameter(torch.zeros(1))
    
    def forward(self, x_flat, edge_flat):
        """x_flat: (nW, N, C), edge_flat: (nW, N, 1)"""
        nW, N, C = x_flat.shape
        
        qkv = self.qkv(x_flat).reshape(nW, N, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Relative position bias
        bias = self.rel_pos[self.rel_index.view(-1)].view(N, N, self.heads)
        bias = bias.permute(2, 0, 1).unsqueeze(0)
        attn = attn + bias
        
        # Edge-guided bias
        edge_bias = self.edge_beta * edge_flat.squeeze(-1).unsqueeze(1).unsqueeze(2)
        attn = attn + edge_bias
        
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(nW, N, C)
        return self.proj(out)


class EdgeGuidedTransformerBlock(nn.Module):
    """Single transformer block with windowed edge-guided attention + FFN."""
    def __init__(self, dim, window_size=8, num_heads=6, mlp_ratio=2.0):
        super().__init__()
        self.win = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowedEdgeAttention(dim, window_size, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
        )
    
    def forward(self, x, edge_map):
        B, C, H, W = x.shape
        win = self.win
        
        pad_h = (win - H % win) % win
        pad_w = (win - W % win) % win
        if pad_h > 0 or pad_w > 0:
            xp = F.pad(x, [0, pad_w, 0, pad_h], mode="reflect")
            ep = F.pad(edge_map, [0, pad_w, 0, pad_h], mode="reflect")
        else:
            xp, ep = x, edge_map
        Hp, Wp = xp.shape[-2:]
        
        x_win = window_partition(xp, win)
        e_win = window_partition(ep, win)
        
        nW_total = x_win.shape[0]
        N = win * win
        x_flat = x_win.view(nW_total, C, N).permute(0, 2, 1)
        e_flat = e_win.view(nW_total, 1, N).permute(0, 2, 1)
        
        shortcut = x_flat
        x_flat = self.norm1(x_flat)
        x_flat = self.attn(x_flat, e_flat)
        x_flat = shortcut + x_flat
        x_flat = x_flat + self.mlp(self.norm2(x_flat))
        
        x_win = x_flat.permute(0, 2, 1).view(nW_total, C, win, win)
        out = window_reverse(x_win, win, Hp, Wp, B)
        
        return out[:, :, :H, :W]


class RSTB(nn.Module):
    """Residual Swin-Transformer Block with gradient checkpointing support."""
    def __init__(self, dim, depth=2, window_size=8, num_heads=6):
        super().__init__()
        self.blocks = nn.ModuleList([
            EdgeGuidedTransformerBlock(dim, window_size, num_heads)
            for _ in range(depth)
        ])
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)
    
    def forward(self, x, edge_map):
        res = x
        for blk in self.blocks:
            x = blk(x, edge_map)
        x = self.conv(x)
        return x + res

In [ ]:
# ============================================================
# 2c. Frequency branch -- FFT-based FRN
# ============================================================

class FRN(nn.Module):
    """Frequency-domain Reconstruction Network using torch.fft.rfft2."""
    def __init__(self, n_bands=4, feat=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_bands * 2, feat, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feat, n_bands * 2, 3, 1, 1),
        )
    
    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            fft_x = torch.fft.rfft2(x, norm="backward")
            fft_cat = torch.cat([fft_x.real, fft_x.imag], dim=1)
            out_fft = self.body(fft_cat)
            real, imag = torch.chunk(out_fft, 2, dim=1)
            spatial = torch.fft.irfft2(
                torch.complex(real, imag), s=x.shape[-2:], norm="backward"
            )
        return spatial

In [ ]:
# ============================================================
# 2d. Guided fusion + PixelShuffle upsampler
# ============================================================

class GuidedFusion(nn.Module):
    def __init__(self, n_bands=4, feat=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(n_bands, feat, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(feat, 1, 3, 1, 1),
        )
    
    def forward(self, x_spatial, x_freq):
        g = torch.sigmoid(self.gate(x_freq))
        fused = g * x_freq + (1 - g) * x_spatial
        return fused, g


class PixelShuffleUpsampler(nn.Module):
    def __init__(self, n_bands, scale, feat=48):
        super().__init__()
        self.expand = nn.Conv2d(n_bands, feat * scale * scale, 3, 1, 1)
        self.ps = nn.PixelShuffle(scale)
        self.out = nn.Conv2d(feat, n_bands, 3, 1, 1)
    
    def forward(self, x):
        return self.out(self.ps(self.expand(x)))

In [ ]:
# ============================================================
# 2e. Full Terra Lens model -- Kaggle-optimized
# ============================================================

class TerraLens(nn.Module):
    """Terra Lens: dual-domain (spatial + frequency) super-resolution network.
    
    Optimized for Kaggle T4x2:
      - dim=48, n_rstb=4, depth_per_rstb=2 (fits in 15GB per GPU)
      - Gradient checkpointing on RSTB blocks
      - Windowed attention (8x8) for O(N) memory
    """
    def __init__(self, n_bands=4, dim=48, depth_per_rstb=2, n_rstb=4,
                 window_size=8, num_heads=6, scale=4, use_checkpointing=True):
        super().__init__()
        self.n_bands = n_bands
        self.scale = scale
        self.use_checkpointing = use_checkpointing
        
        self.edge = SobelEdge(n_bands)
        self.shallow = nn.Conv2d(n_bands, dim, 3, 1, 1)
        self.rstbs = nn.ModuleList([
            RSTB(dim, depth_per_rstb, window_size, num_heads)
            for _ in range(n_rstb)
        ])
        self.spatial_out = nn.Conv2d(dim, n_bands, 3, 1, 1)
        self.frn = FRN(n_bands, feat=32)
        self.fusion = GuidedFusion(n_bands)
        self.upsampler = PixelShuffleUpsampler(n_bands, scale, feat=dim)
    
    def forward(self, lr):
        edge_map = self.edge(lr)
        
        feat = self.shallow(lr)
        for rstb in self.rstbs:
            if self.use_checkpointing and self.training:
                feat = checkpoint(rstb, feat, edge_map, use_reentrant=False)
            else:
                feat = rstb(feat, edge_map)
        x_spatial = self.spatial_out(feat)
        
        x_freq = self.frn(lr)
        fused, gate = self.fusion(x_spatial, x_freq)
        
        sr = self.upsampler(fused)
        sr = sr + F.interpolate(lr, scale_factor=self.scale, mode="bicubic", align_corners=False)
        return sr


# Build and inspect
model = TerraLens(
    n_bands=4, dim=48, depth_per_rstb=2, n_rstb=4,
    window_size=8, num_heads=6, scale=SCALE, use_checkpointing=True
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTerra Lens model:")
print(f"  Total parameters:     {n_params:,} ({n_params/1e6:.2f}M)")
print(f"  Trainable parameters: {n_trainable:,} ({n_trainable/1e6:.2f}M)")

# DataParallel for dual T4
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"  Using DataParallel across {n_gpus} GPUs")

# Quick shape test
with torch.no_grad():
    test_lr = torch.randn(1, 4, 32, 32, device=DEVICE)
    test_sr = model(test_lr)
    print(f"  Shape test: LR {list(test_lr.shape)} -> SR {list(test_sr.shape)}")
    del test_lr, test_sr
    if USE_CUDA: torch.cuda.empty_cache()

## 3. Loss Functions & Metrics

In [ ]:
# ============================================================
# 3. Loss: Charbonnier + FFT frequency-domain + SAM
# ============================================================

class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    
    def forward(self, x, y):
        return torch.mean(torch.sqrt((x - y) ** 2 + self.eps ** 2))


def spectral_angle_mapper(sr, hr, eps=1e-8):
    dot = (sr * hr).sum(dim=1)
    n1 = sr.norm(dim=1)
    n2 = hr.norm(dim=1)
    cos = (dot / (n1 * n2 + eps)).clamp(-1 + 1e-7, 1 - 1e-7)
    return torch.acos(cos).mean() * 180.0 / math.pi


def fft_loss(sr, hr):
    with torch.autocast(device_type=sr.device.type, enabled=False):
        sr_fft = torch.fft.rfft2(sr.float(), norm="backward")
        hr_fft = torch.fft.rfft2(hr.float(), norm="backward")
        return F.l1_loss(torch.abs(sr_fft), torch.abs(hr_fft))


charbonnier = CharbonnierLoss()

def terra_lens_loss(sr, hr, w_freq=0.1, w_sam=0.05):
    l_pix = charbonnier(sr, hr)
    l_freq = fft_loss(sr, hr)
    l_sam = spectral_angle_mapper(sr.float(), hr.float()) / 180.0
    total = l_pix + w_freq * l_freq + w_sam * l_sam
    return total, {
        "pix": l_pix.item(),
        "freq": l_freq.item(),
        "sam": l_sam.item() * 180.0,
    }


def calculate_psnr(sr, hr, max_val=1.0):
    mse = F.mse_loss(sr.float(), hr.float())
    return 10 * torch.log10(max_val ** 2 / (mse + 1e-10))

def calculate_sam(sr, hr):
    return spectral_angle_mapper(sr.float(), hr.float())

def ndvi(x, red_idx=2, nir_idx=3, eps=1e-6):
    red, nir = x[:, red_idx], x[:, nir_idx]
    return (nir - red) / (nir + red + eps)

def ndvi_consistency(sr, hr):
    return F.l1_loss(ndvi(sr), ndvi(hr))

print("Loss functions and metrics ready.")

## 4. Early Stopping & Training Infrastructure

In [ ]:
# ============================================================
# 4. Early stopping + training utilities
# ============================================================

CKPT_DIR = WORKING_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.01, mode='max', verbose=True, save_path=None):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.save_path = save_path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_weights = None
    
    def _snapshot(self, model):
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        self.best_weights = {k: v.detach().cpu().clone() for k, v in state.items()}
        if self.save_path:
            torch.save(self.best_weights, self.save_path)
    
    def set_baseline(self, score, model):
        self.best_score = score
        self._snapshot(model)
    
    def __call__(self, current_score, model):
        if self.best_score is None:
            self.best_score = current_score
            self._snapshot(model)
            return True
        
        improved = (current_score - self.best_score) > self.min_delta if self.mode == 'max' else (self.best_score - current_score) > self.min_delta
        
        if improved:
            if self.verbose:
                print(f"  [ES] Improved ({self.best_score:.4f} -> {current_score:.4f}). Saved.")
            self.best_score = current_score
            self._snapshot(model)
            self.counter = 0
            return True
        
        self.counter += 1
        if self.verbose:
            print(f"  [ES] No improvement for {self.counter}/{self.patience} epochs.")
        if self.counter >= self.patience:
            self.early_stop = True
        return False
    
    def restore_best_weights(self, model):
        if self.best_weights is not None:
            target = model.module if hasattr(model, 'module') else model
            target.load_state_dict({k: v.to(DEVICE) for k, v in self.best_weights.items()})


@torch.no_grad()
def bicubic_baseline_psnr(loader, scale):
    total_psnr = 0.0
    n = 0
    for lr_v, hr_v in loader:
        up = F.interpolate(lr_v.to(DEVICE), scale_factor=scale,
                          mode="bicubic", align_corners=False).clamp(0, 1)
        total_psnr += calculate_psnr(up, hr_v.to(DEVICE)).item()
        n += 1
        if n >= 20:
            break
    return total_psnr / max(1, n)


bicubic_psnr = bicubic_baseline_psnr(val_loader, SCALE)
print(f"\nBicubic baseline Val PSNR: {bicubic_psnr:.2f} dB (model must beat this)")

## 5. Stage 1 -- Full Joint Training

In [ ]:
# ============================================================
# 5a. Generic training stage
# ============================================================

def run_stage(model, train_loader, val_loader, optimizer, scaler, scheduler,
              max_epochs, stage_name, accum_steps=2, patience=10,
              baseline=None, save_path=None):
    print(f"\n{'='*70}")
    print(f"  {stage_name} -- Max Epochs: {max_epochs}, Patience: {patience}")
    print(f"{'='*70}")
    
    early_stopper = EarlyStopping(
        patience=patience, min_delta=0.01, mode='max',
        verbose=True, save_path=save_path
    )
    if baseline is not None:
        early_stopper.set_baseline(baseline, model)
    
    for epoch in range(1, max_epochs + 1):
        model.train()
        t0 = time.time()
        train_loss = 0.0
        train_parts = {"pix": 0.0, "freq": 0.0, "sam": 0.0}
        optimizer.zero_grad(set_to_none=True)
        
        for i, (lr_b, hr_b) in enumerate(train_loader):
            lr_b = lr_b.to(DEVICE, non_blocking=True)
            hr_b = hr_b.to(DEVICE, non_blocking=True)
            
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                sr_b = model(lr_b)
                loss, parts = terra_lens_loss(sr_b.float(), hr_b)
                loss = loss / accum_steps
            
            scaler.scale(loss).backward()
            train_loss += loss.item() * accum_steps
            for k in train_parts:
                train_parts[k] += parts.get(k, 0.0)
            
            if (i + 1) % accum_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        
        if scheduler is not None:
            scheduler.step()
        
        if USE_CUDA:
            torch.cuda.empty_cache()
        gc.collect()
        
        # Validation
        model.eval()
        val_psnr = 0.0
        val_sam = 0.0
        val_ndvi = 0.0
        n_val = 0
        with torch.no_grad():
            for lr_v, hr_v in val_loader:
                lr_v = lr_v.to(DEVICE, non_blocking=True)
                hr_v = hr_v.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    sr_v = model(lr_v)
                sr_v = sr_v.float().clamp(0, 1)
                val_psnr += calculate_psnr(sr_v, hr_v).item()
                val_sam += calculate_sam(sr_v, hr_v).item()
                val_ndvi += ndvi_consistency(sr_v, hr_v).item()
                n_val += 1
        
        avg_psnr = val_psnr / max(1, n_val)
        avg_sam = val_sam / max(1, n_val)
        avg_ndvi = val_ndvi / max(1, n_val)
        n_steps = max(1, len(train_loader))
        
        elapsed = time.time() - t0
        lr_current = optimizer.param_groups[0]['lr']
        
        mem_str = ""
        if USE_CUDA:
            mem_used = torch.cuda.max_memory_allocated() / 1e9
            mem_str = f" | VRAM: {mem_used:.1f}GB"
            torch.cuda.reset_peak_memory_stats()
        
        print(
            f"[{stage_name}|E{epoch:02d}/{max_epochs}] "
            f"Loss: {train_loss/n_steps:.4f} "
            f"(pix={train_parts['pix']/n_steps:.4f} freq={train_parts['freq']/n_steps:.4f} "
            f"SAM={train_parts['sam']/n_steps:.1f} deg) | "
            f"Val PSNR: {avg_psnr:.2f}dB SAM: {avg_sam:.1f} deg NDVI_L1: {avg_ndvi:.4f} | "
            f"LR: {lr_current:.1e} | {elapsed:.0f}s{mem_str}",
            flush=True
        )
        
        early_stopper(avg_psnr, model)
        if early_stopper.early_stop:
            print(f"\n--> Early stopping triggered at epoch {epoch}.")
            break
    
    early_stopper.restore_best_weights(model)
    print(f"\n--> Best {stage_name} PSNR: {early_stopper.best_score:.2f} dB")
    return early_stopper.best_score

In [ ]:
# ============================================================
# 5b. Stage 1 execution
# ============================================================

EPOCHS_STAGE1 = 60
LR_STAGE1 = 3e-4

opt1 = torch.optim.AdamW(model.parameters(), lr=LR_STAGE1, weight_decay=1e-4)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EPOCHS_STAGE1, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best1 = run_stage(
    model, train_loader, val_loader, opt1, scaler, sched1,
    max_epochs=EPOCHS_STAGE1,
    stage_name="Stage 1",
    accum_steps=ACCUM_STEPS,
    patience=12,
    save_path=str(CKPT_DIR / "terralens_stage1_best.pt"),
)

print(f"\nStage 1 complete. Best PSNR: {best1:.2f} dB (bicubic: {bicubic_psnr:.2f} dB, gain: {best1 - bicubic_psnr:+.2f} dB)")

## 6. Stage 2 -- Fine-tune with lower LR

In [ ]:
# ============================================================
# 6. Stage 2 -- Fine-tuning with lower LR
# ============================================================

EPOCHS_STAGE2 = 25
LR_STAGE2 = 5e-5

opt2 = torch.optim.AdamW(model.parameters(), lr=LR_STAGE2, weight_decay=1e-5)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_STAGE2, eta_min=1e-7)

best_psnr = run_stage(
    model, train_loader, val_loader, opt2, scaler, sched2,
    max_epochs=EPOCHS_STAGE2,
    stage_name="Stage 2",
    accum_steps=ACCUM_STEPS,
    patience=8,
    baseline=best1,
    save_path=str(CKPT_DIR / "terralens_best.pt"),
)

# Save final model
final_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
torch.save(final_state, str(CKPT_DIR / "terralens_final.pt"))

print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE")
print(f"  Final Best PSNR:     {best_psnr:.2f} dB")
print(f"  Bicubic Baseline:    {bicubic_psnr:.2f} dB")
print(f"  Gain over bicubic:   {best_psnr - bicubic_psnr:+.2f} dB")
print(f"  Checkpoints saved to: {CKPT_DIR}")
print(f"{'='*70}")

## 7. Final Evaluation

In [ ]:
# ============================================================
# 7. Final evaluation on validation set
# ============================================================

model.eval()
eval_psnr = []
eval_sam = []
eval_ndvi = []

with torch.no_grad():
    for lr_v, hr_v in val_loader:
        lr_v = lr_v.to(DEVICE)
        hr_v = hr_v.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            sr_v = model(lr_v)
        sr_v = sr_v.float().clamp(0, 1)
        eval_psnr.append(calculate_psnr(sr_v, hr_v).item())
        eval_sam.append(calculate_sam(sr_v, hr_v).item())
        eval_ndvi.append(ndvi_consistency(sr_v, hr_v).item())

results = {
    "psnr_db": float(np.mean(eval_psnr)),
    "sam_degrees": float(np.mean(eval_sam)),
    "ndvi_l1": float(np.mean(eval_ndvi)),
    "bicubic_baseline_psnr": bicubic_psnr,
    "gain_over_bicubic_db": float(np.mean(eval_psnr)) - bicubic_psnr,
    "dataset": "s2-naip-5k-pairs-rgb-nir",
    "scale": SCALE,
    "n_train": len(train_ds),
    "n_val": len(val_ds),
    "patch_size": PATCH_SIZE,
    "model_params_M": n_params / 1e6,
}

print("\n" + "=" * 50)
print("  FINAL EVALUATION RESULTS")
print("=" * 50)
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")
print("=" * 50)

with open(str(WORKING_DIR / "pipelineA_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {WORKING_DIR / 'pipelineA_results.json'}")

## 8. Visual Sanity Check

In [ ]:
# ============================================================
# 8. Visual comparison: LR -> Bicubic -> Terra Lens SR
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

model.eval()
lr_b, hr_b = next(iter(val_loader))
lr_b, hr_b = lr_b.to(DEVICE), hr_b.to(DEVICE)

with torch.no_grad():
    with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
        sr_b = model(lr_b)
    sr_b = sr_b.float().clamp(0, 1)

def to_rgb(t, percentile_stretch=True):
    """Convert 4-band tensor to RGB for display."""
    rgb = t[[2, 1, 0]].clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    if percentile_stretch:
        for c in range(3):
            lo, hi = np.percentile(rgb[:, :, c], [2, 98])
            if hi - lo > 1e-6:
                rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo), 0, 1)
    return rgb

n_show = min(4, lr_b.shape[0])
fig, axes = plt.subplots(n_show, 3, figsize=(15, 5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for i in range(n_show):
    bic = F.interpolate(lr_b[i:i+1], size=hr_b.shape[-2:], mode="bicubic", align_corners=False).clamp(0, 1)
    
    axes[i, 0].imshow(to_rgb(bic[0]))
    axes[i, 0].set_title("Bicubic (baseline)", fontsize=11)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(to_rgb(sr_b[i]))
    axes[i, 1].set_title("Terra Lens SR", fontsize=11, color="green", fontweight="bold")
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(to_rgb(hr_b[i]))
    axes[i, 2].set_title("Ground Truth (NAIP)", fontsize=11)
    axes[i, 2].axis("off")

fig.suptitle("Pipeline A -- S2-NAIP Super-Resolution Results", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(str(WORKING_DIR / "pipelineA_visual_samples.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved visual samples to {WORKING_DIR / 'pipelineA_visual_samples.png'}")

In [ ]:
# ============================================================
# 9. NDVI comparison
# ============================================================

fig, axes = plt.subplots(n_show, 3, figsize=(18, 5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for i in range(n_show):
    bic = F.interpolate(lr_b[i:i+1], size=hr_b.shape[-2:], mode="bicubic", align_corners=False).clamp(0, 1)
    
    bic_ndvi = ndvi(bic)[0].cpu().numpy()
    sr_ndvi = ndvi(sr_b[i:i+1])[0].cpu().numpy()
    hr_ndvi = ndvi(hr_b[i:i+1])[0].cpu().numpy()
    
    vmin, vmax = -0.2, 0.8
    
    im0 = axes[i, 0].imshow(bic_ndvi, cmap="RdYlGn", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title("NDVI -- Bicubic")
    axes[i, 0].axis("off")
    plt.colorbar(im0, ax=axes[i, 0], fraction=0.046)
    
    im1 = axes[i, 1].imshow(sr_ndvi, cmap="RdYlGn", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title("NDVI -- Terra Lens SR", color="green", fontweight="bold")
    axes[i, 1].axis("off")
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046)
    
    im2 = axes[i, 2].imshow(hr_ndvi, cmap="RdYlGn", vmin=vmin, vmax=vmax)
    axes[i, 2].set_title("NDVI -- Ground Truth")
    axes[i, 2].axis("off")
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

fig.suptitle("NDVI Comparison -- Pipeline A", fontsize=14, fontweight="bold")
plt.tight_layout()
fig.savefig(str(WORKING_DIR / "pipelineA_ndvi_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved NDVI comparison to {WORKING_DIR / 'pipelineA_ndvi_comparison.png'}")

In [ ]:
# ============================================================
# 10. Summary
# ============================================================
print("\n" + "=" * 70)
print("  PIPELINE A -- TRAINING COMPLETE")
print("=" * 70)
print(f"  Dataset:             S2-NAIP 5k pairs (RGB + NIR)")
print(f"  Layout:              {layout_mode}")
print(f"  Scale factor:        {SCALE}x")
print(f"  Model:               Terra Lens (dim=48, 4 RSTB, windowed attention)")
print(f"  Parameters:          {n_params/1e6:.2f}M")
print(f"  Train samples:       {len(train_ds)}")
print(f"  Val samples:         {len(val_ds)}")
print(f"  Bicubic baseline:    {bicubic_psnr:.2f} dB")
print(f"  Final best PSNR:     {best_psnr:.2f} dB (+{best_psnr - bicubic_psnr:.2f} dB)")
print(f"  GPUs used:           {n_gpus}")
if USE_CUDA:
    for i in range(torch.cuda.device_count()):
        print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"  AMP dtype:           {AMP_DTYPE}")
print(f"  Checkpoints:         {CKPT_DIR}")
print(f"\n  Files saved:")
for f in sorted(CKPT_DIR.glob("*.pt")):
    print(f"    {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
for f in sorted(WORKING_DIR.glob("pipelineA_*")):
    print(f"    {f.name}")
print("=" * 70)

## 11. Zip Outputs for Download

In [ ]:
# ============================================================
# 11. Create a zip archive of strictly weights and results
# ============================================================
import zipfile
import os

zip_name = 'terralens_pipelineA_outputs.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add checkpoints
    if CKPT_DIR.exists():
        for f in CKPT_DIR.glob('*.pt'):
            zipf.write(f, arcname=f'checkpoints/{f.name}')
    # Add evaluation results and images
    for f in WORKING_DIR.glob('pipelineA_*'):
        zipf.write(f, arcname=f.name)

print(f'Created {zip_name} containing strictly checkpoints and results!')
print('You can now download this file from the Kaggle Data menu in the right sidebar.')
